1.实现池化层的正向传播

X - （H, W）
pool_size - 一个元组 (p_h, p_w),表示池化窗口的高度和宽度
mode - 'max'表示最大池化， avg-平均池化
Y - 池化后，(H - p_h + 1, W - p_w + 1)

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

def pool2d(X, pool_size, mode='max'):
    p_h, p_w = pool_size
    Y = torch.zeros((X.shape[0] - p_h + 1, X.shape[1] - p_w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i, j] = X[i:i + p_h, j:j + p_w].max()
            elif mode == 'avg':
                Y[i, j] = X[i:i + p_h, j:j + p_w].mean()
    return Y

2. 验证二维最大池化层的输出

In [2]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
pool2d(X, (2, 2))

tensor([[4., 5.],
        [7., 8.]])

3. 验证平均池化层

In [3]:
pool2d(X, (2, 2), 'avg')

tensor([[2., 3.],
        [5., 6.]])

4. 调用层实现填充和步幅
创建了一个形状为 (1, 1, 4, 4) 的张量，这是一个 4 维张量，各维度含义为：
（batch_size, channels, height, width）
X = [[[[ 0.,  1.,  2.,  3.],
     [ 4.,  5.,  6.,  7.],
     [ 8.,  9., 10., 11.],
     [12., 13., 14., 15.]]]]

In [4]:
X = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))
X

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]]]])

5.深度学习框架中步幅与池化窗口大小相同 - 池化时窗口不重叠
只返回前面的3x3

In [5]:
# 3x3的窗口
pool2d = nn.MaxPool2d(3)
pool2d(X)

tensor([[[[10.]]]])

6. 填充和步幅可以手动设定

In [6]:
pool2d = nn.MaxPool2d(3, padding=1, stride=2)
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]]]])

7. 可设定一个任意大小的池化矩阵窗口，并分别设定填充和步幅的高度和宽度
stride(2, 3) 
池化窗口在高度方向每次向下移动 2 个像素。
池化窗口在宽度方向每次向右移动 3 个像素。

In [7]:
pool2d = nn.MaxPool2d((2, 3))
pool2d(X)

tensor([[[[ 6.],
          [14.]]]])

8. 池化层在每个输入通道上单独运算
cat是等维拼接


In [8]:
# 1x2x4x4
X = torch.cat((X, X + 1), 1)
X

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]],

         [[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])

In [9]:
pool2d = nn.MaxPool2d(3, padding=1, stride=2)
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]],

         [[ 6.,  8.],
          [14., 16.]]]])